# Generate Music Embeddings on Google Colab

This notebook generates semantic embeddings for music plays using state-of-the-art sentence transformers.

**Models available:**
- `all-mpnet-base-v2` (420M params, 768 dims) - Best quality/speed tradeoff
- `all-MiniLM-L12-v2` (33M params, 384 dims) - Faster, smaller
- `multi-qa-mpnet-base-dot-v1` (420M params, 768 dims) - Optimized for semantic search

**Requirements:**
1. Upload `enriched_plays_full.csv` to Colab
2. Enable GPU runtime (Runtime → Change runtime type → T4 GPU)
3. Run all cells

In [ ]:
# Install dependencies
!pip install -q sentence-transformers pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import time
import torch

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Configuration
INPUT_FILE = 'enriched_plays_full.csv'  # Upload this file first
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'  # Best quality
# MODEL_NAME = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'  # Alternative: optimized for search
# MODEL_NAME = 'sentence-transformers/all-MiniLM-L12-v2'  # Alternative: faster/smaller

BATCH_SIZE = 128 if device == 'cuda' else 32  # Larger batches for GPU
OUTPUT_EMBEDDINGS = 'embeddings.npy'
OUTPUT_METADATA = 'metadata.csv'

print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Load data
print("Loading data...")
df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df):,} plays")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample enriched text:")
print(df['enriched_text'].iloc[0][:200] + "...")

# Extract texts for embedding
texts = df['enriched_text'].fillna('').tolist()
print(f"\nPrepared {len(texts):,} texts for embedding")

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=device)

embedding_dim = model.get_sentence_embedding_dimension()
print(f"✓ Model loaded")
print(f"  Embedding dimensions: {embedding_dim}")
print(f"  Max sequence length: {model.max_seq_length}")

In [ ]:
# Generate embeddings
print(f"\nGenerating embeddings for {len(texts):,} texts...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}")

start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

elapsed = time.time() - start_time
texts_per_sec = len(texts) / elapsed

print(f"\n✓ Generated {len(embeddings):,} embeddings")
print(f"  Time: {elapsed:.1f}s ({texts_per_sec:.1f} texts/sec)")
print(f"  Shape: {embeddings.shape}")
print(f"  Size: {embeddings.nbytes / 1e6:.1f} MB")

In [ ]:
# Save embeddings
print(f"\nSaving embeddings to {OUTPUT_EMBEDDINGS}...")
np.save(OUTPUT_EMBEDDINGS, embeddings)
print(f"✓ Saved embeddings ({embeddings.nbytes / 1e6:.1f} MB)")

# Save metadata (subset of columns for efficient loading)
metadata_columns = [
    'id', 'artist', 'artist_ids', 'song', 'recording_id', 
    'album', 'release_id', 'airdate', 'labels', 'rotation_status'
]
available_columns = [col for col in metadata_columns if col in df.columns]

print(f"\nSaving metadata to {OUTPUT_METADATA}...")
df[available_columns].to_csv(OUTPUT_METADATA, index=False)
print(f"✓ Saved metadata ({len(available_columns)} columns)")

print(f"\n{'='*80}")
print("COMPLETE!")
print(f"{'='*80}")
print(f"\nGenerated files:")
print(f"  1. {OUTPUT_EMBEDDINGS} - Numpy array ({embeddings.shape[0]:,} x {embeddings.shape[1]})")
print(f"  2. {OUTPUT_METADATA} - Metadata CSV ({len(df):,} rows)")
print(f"\nDownload both files and use them in your application!")

## Test Search (Optional)

Test the embeddings with a sample search query:

In [ ]:
# Test search
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=10):
    """Search for similar music using semantic similarity."""
    # Encode query
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    
    # Compute similarities (already normalized, so dot product = cosine similarity)
    similarities = embeddings @ query_embedding.T
    
    # Get top results
    top_indices = np.argsort(similarities.flatten())[-top_k:][::-1]
    
    print(f"\nQuery: '{query}'\n")
    print(f"{'='*80}")
    
    for rank, idx in enumerate(top_indices, 1):
        score = similarities[idx][0]
        play = df.iloc[idx]
        print(f"{rank:2d}. [{score:.3f}] {play['artist']} - {play['song']}")
        if 'album' in play and pd.notna(play['album']):
            print(f"    Album: {play['album']}")
        print()

# Example searches
search("psychedelic folk rock")
search("upbeat dance electronic")
search("melancholic indie with female vocals")